# ASL fingerspelling: ablations on GPU

Warm-starts every variant from the v5 weights on the Hub and scores each on the same signer-independent split with beam search. Then fits the character LM on training-split phrases only, tunes its weight on validation, and reports test once.

In [ ]:
BRANCH = "phase-4-accuracy"
!git clone -q --branch {BRANCH} https://github.com/sohenpatel22/ASL-Fingerspelling-Recognition.git /tmp/repo
%cd /tmp/repo
!git log --oneline -1
!pip install -q -e .
import torch
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
import os
import zipfile
from pathlib import Path

def find_dir(name, root="/kaggle/input", max_depth=5):
    for base, dirs, _ in os.walk(root):
        depth = base[len(root):].count(os.sep)
        if name in dirs:
            return Path(base, name)
        if depth >= max_depth:
            dirs[:] = []
    raise FileNotFoundError(name)

COMP = next(p for p in (Path("/kaggle/input/competitions/asl-fingerspelling"), Path("/kaggle/input/asl-fingerspelling")) if p.exists())
try:
    NPY_TRAIN, NPY_SUPP = find_dir("npy_train"), find_dir("npy_supp")
except FileNotFoundError:
    # the preprocess output can be mounted as one zip instead of a folder tree
    archives = [p for p in Path("/kaggle/input").rglob("*.zip") if "competitions" not in str(p)]
    print("unzipping", archives)
    for archive in archives:
        with zipfile.ZipFile(archive) as z:
            z.extractall("/tmp/npy")
    NPY_TRAIN, NPY_SUPP = find_dir("npy_train", "/tmp/npy"), find_dir("npy_supp", "/tmp/npy")
print(len(list(NPY_TRAIN.glob("*.npy"))), "train files,", len(list(NPY_SUPP.glob("*.npy"))), "supplemental files")
DATA = f"data.comp_dir={COMP} data.npy_train={NPY_TRAIN} data.npy_supp={NPY_SUPP}"
print(DATA)

In [ ]:
SMOKE = False   # True = quick end-to-end check of every stage (about 10 minutes)
if SMOKE:
    EXTRA = "train.epochs=1 train.eval_samples=200 data.use_supplemental=false"
    ONLY, N_VAL, N_TEST = "--only a0_baseline,a5_all --val-samples 100 --test-samples 100", 100, 100
else:
    EXTRA, ONLY, N_VAL, N_TEST = "", "", 1000, 3000

!asl-ablate --config configs/finetune_v5.yaml --out-dir /kaggle/working/runs {DATA} {EXTRA} {ONLY}

In [ ]:
import json

assert Path("/kaggle/working/runs/results.json").exists(), "asl-ablate failed: read the output of the cell above"
results = json.load(open("/kaggle/working/runs/results.json"))
best = min(results, key=lambda n: results[n]["val_beam_cer"])   # choose on validation, never on test
print("best variant on validation:", best)
print(open("/kaggle/working/runs/results.md").read())

In [ ]:
overrides = " ".join(results[best]["overrides"])
!asl-lm --config configs/finetune_v5.yaml --out /kaggle/working/char_lm.json.gz {DATA} {EXTRA}
!asl-eval --checkpoint /kaggle/working/runs/{best}/best.pth --config configs/finetune_v5.yaml --split val --max-samples {N_VAL} --sample-seed 0 --sweep-lm-weights 0,0.1,0.2,0.3,0.5 --out /kaggle/working/lm_sweep.json {DATA} decode.lm_path=/kaggle/working/char_lm.json.gz {overrides}
assert Path("/kaggle/working/lm_sweep.json").exists(), "the language model sweep failed: read the output above"

In [ ]:
sweep = json.load(open("/kaggle/working/lm_sweep.json"))
weight = sweep["best_lm_weight"]
print("lm weight chosen on validation:", weight)
!asl-eval --checkpoint /kaggle/working/runs/{best}/best.pth --config configs/finetune_v5.yaml --split test --max-samples {N_TEST} --sample-seed 0 --out /kaggle/working/final_test_lm.json --details-out /kaggle/working/final_test_lm_details.json {DATA} decode.lm_path=/kaggle/working/char_lm.json.gz decode.lm_weight={weight} {overrides}
print(open("/kaggle/working/final_test_lm.json").read())